In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("raw_expense_data_v2.csv")
print(df.shape)
print(df.isnull().sum())
print(df.dtypes)

(2538, 14)
Transaction ID       0
Date                57
Type                 0
Category           174
Subcategory        558
Amount               0
Payment Mode       338
Account              0
Recurring            0
Necessity          384
Merchant           222
Budget Limit       754
Description          0
Tags              1259
dtype: int64
Transaction ID     object
Date               object
Type               object
Category           object
Subcategory        object
Amount             object
Payment Mode       object
Account            object
Recurring          object
Necessity          object
Merchant           object
Budget Limit      float64
Description        object
Tags               object
dtype: object


In [2]:
df["Amount"] = (
    df["Amount"].astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [3]:
df["Category"] = df["Category"].str.strip().str.title()

In [4]:
df["Category"] = df["Category"].fillna("Uncategorized")
df["Subcategory"] = df["Subcategory"].fillna("General")
df["Payment Mode"] = df["Payment Mode"].fillna("Unknown")
df["Merchant"] = df["Merchant"].fillna("Unknown Merchant")
df["Tags"] = df["Tags"].fillna("None")

In [5]:
df.loc[(df["Type"] == "Expense") & (df["Necessity"].isna()), "Necessity"] = "Discretionary"
df["Necessity"] = df["Necessity"].fillna("N/A")

In [6]:
df["Budget Limit"] = df["Budget Limit"].fillna(0)

In [7]:
before = len(df)
df = df.dropna(subset=["Date"])
print(f"Dropped {before - len(df)} rows")

Dropped 57 rows


In [8]:
df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["MonthName"] = df["Date"].dt.strftime("%b")
df["Quarter"] = "Q" + df["Date"].dt.quarter.astype(str)
df["Weekday"] = df["Date"].dt.day_name()

In [9]:
df[["Date","Year","Month","MonthName","Quarter","Weekday"]].head()

,Date,Year,Month,MonthName,Quarter,Weekday
0,2024-05-31,2024,5,May,Q2,Friday
1,2025-05-03,2025,5,May,Q2,Saturday
2,2025-05-31,2025,5,May,Q2,Saturday
3,2025-09-19,2025,9,Sep,Q3,Friday
4,2025-10-18,2025,10,Oct,Q4,Saturday


In [10]:
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows")

Dropped 37 duplicate rows


In [11]:
q99 = df.loc[df["Type"] == "Expense", "Amount"].quantile(0.99)
df["Is_Outlier"] = (df["Type"] == "Expense") & (df["Amount"] > q99)
print(f"99th percentile: {q99:.0f}, flagged: {df['Is_Outlier'].sum()}")

99th percentile: 7983, flagged: 21


In [12]:
assert (df["Amount"] >= 0).all()
assert df["Transaction ID"].is_unique

In [13]:
df.to_csv("cleaned_expense_data_v2.csv", index=False)

In [14]:
import os
print(os.getcwd())
print(os.path.exists("cleaned_expense_data_v2.csv"))

C:\Users\gokul\OneDrive\Desktop\VScode\DAVIS
True


In [15]:
print(df.shape)
print(df.isnull().sum().sum())
df.head(10)

(2444, 20)
0


,Transaction ID,Date,Type,Category,Subcategory,Amount,Payment Mode,Account,Recurring,Necessity,Merchant,Budget Limit,Description,Tags,Year,Month,MonthName,Quarter,Weekday,Is_Outlier
0,TXN100576,2024-05-31,Expense,Healthcare,Pharmacy,1328.28,Net Banking,Credit Card A/C,Yes,Essential,Swiggy,3000.0,Healthcare transaction #576,review,2024,5,May,Q2,Friday,False
1,TXN100431,2025-05-03,Expense,Uncategorized,General,2378.73,Credit Card,Primary Savings,No,Discretionary,BESCOM,0.0,Misc transaction #431,one-time,2025,5,May,Q2,Saturday,False
2,TXN101277,2025-05-31,Expense,Uncategorized,General,5620.14,UPI,Credit Card A/C,No,Discretionary,BESCOM,0.0,Misc transaction #1277,review,2025,5,May,Q2,Saturday,False
3,TXN102473,2025-09-19,Expense,Utilities,Internet,2317.81,Net Banking,Credit Card A/C,Yes,Essential,Employer,3000.0,Utilities transaction #2473,review,2025,9,Sep,Q3,Friday,False
4,TXN100934,2025-10-18,Expense,Food,Dining Out,866.81,Credit Card,Primary Savings,Yes,Discretionary,Swiggy,0.0,food transaction #934,review,2025,10,Oct,Q4,Saturday,False
5,TXN100810,2024-02-23,Expense,Uncategorized,General,3925.66,Cash,Credit Card A/C,Yes,Discretionary,Amazon,0.0,Misc transaction #810,review,2024,2,Feb,Q1,Friday,False
6,TXN100368,2024-12-05,Income,Salary,General,23798.76,Bank Transfer,Salary Account,Yes,N/A,Employer,0.0,Salary transaction #368,reimbursable,2024,12,Dec,Q4,Thursday,False
7,TXN100209,2024-10-08,Expense,Utilities,Electricity,5597.98,Cash,Salary Account,Yes,Essential,Zomato,3000.0,Utilities transaction #209,reimbursable,2024,10,Oct,Q4,Tuesday,False
8,TXN101348,2024-06-11,Income,Gift,General,23504.56,Bank Transfer,Salary Account,No,N/A,Zomato,0.0,Gift transaction #1348,None,2024,6,Jun,Q2,Tuesday,False
9,TXN102062,2024-11-11,Income,Investment,General,35561.41,Bank Transfer,Salary Account,No,N/A,Local Store,0.0,Investment transaction #2062,review,2024,11,Nov,Q4,Monday,False
